# General AI-Powered Intrusion Detection System (IDS)

## Objective
This notebook develops a general AI-driven IDS using the CICIDS2017 dataset. 
This model is designed as a broad IDS capable of detecting and classifying multiple attack types.

The system performs:
1. **Binary detection**: classify traffic as BENIGN or ATTACK
2. **Attack classification**: classify malicious traffic into broader attack categories

Models used:
- Random Forest for binary detection
- Isolation Forest for anomaly detection
- Random Forest for multiclass attack classification

In [49]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [50]:
all_files = [
    r"C:\Users\silva\Documents\ai-powered-ids\ml_detection\datasets\CICIDS2017\Monday-WorkingHours.pcap_ISCX.csv",
    r"C:\Users\silva\Documents\ai-powered-ids\ml_detection\datasets\CICIDS2017\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    r"C:\Users\silva\Documents\ai-powered-ids\ml_detection\datasets\CICIDS2017\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

dfs = [pd.read_csv(f) for f in all_files]
df_all = pd.concat(dfs, ignore_index=True)

print("Combined shape:", df_all.shape)

Combined shape: (1042130, 79)


In [51]:
df_all.columns = df_all.columns.str.strip()
df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
df_all.dropna(inplace=True)

print("Cleaned shape:", df_all.shape)

Cleaned shape: (1041288, 79)


In [52]:
print(df_all["Label"].value_counts())

Label
BENIGN      754459
PortScan    158804
DDoS        128025
Name: count, dtype: int64


In [53]:
df_all["binary_label"] = df_all["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

print(df_all["binary_label"].value_counts())

binary_label
0    754459
1    286829
Name: count, dtype: int64


In [54]:
def map_attack_label(label):
    label = label.strip()

    if label == "BENIGN":
        return "BENIGN"
    elif "DDoS" in label:
        return "DDoS"
    elif "PortScan" in label:
        return "PortScan"
    elif "Patator" in label:
        return "BruteForce"
    elif "Web Attack" in label:
        return "WebAttack"
    elif "DoS" in label:
        return "DoS"
    elif "Bot" in label:
        return "Bot"
    elif "Infiltration" in label:
        return "Infiltration"
    elif "Heartbleed" in label:
        return "Heartbleed"
    else:
        return "Other"

In [55]:
df_all["attack_type"] = df_all["Label"].apply(map_attack_label)

print(df_all["attack_type"].value_counts())

attack_type
BENIGN      754459
PortScan    158804
DDoS        128025
Name: count, dtype: int64


In [56]:
features = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Flow Bytes/s",
    "Flow Packets/s"
]

In [57]:
df_train, df_test = train_test_split(
    df_all,
    test_size=0.2,
    random_state=42,
    stratify=df_all["attack_type"]
)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (833030, 81)
Test shape: (208258, 81)


In [58]:
print("Train attack types:")
print(df_train["attack_type"].value_counts())

print("\nTest attack types:")
print(df_test["attack_type"].value_counts())

Train attack types:
attack_type
BENIGN      603567
PortScan    127043
DDoS        102420
Name: count, dtype: int64

Test attack types:
attack_type
BENIGN      150892
PortScan     31761
DDoS         25605
Name: count, dtype: int64


In [59]:
X_train_bin = df_train[features].copy()
y_train_bin = df_train["binary_label"].copy()

X_test_bin = df_test[features].copy()
y_test_bin = df_test["binary_label"].copy()

print("Binary train shape:", X_train_bin.shape)
print("Binary test shape:", X_test_bin.shape)

Binary train shape: (833030, 8)
Binary test shape: (208258, 8)


In [60]:
rf_binary = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_binary.fit(X_train_bin, y_train_bin)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [61]:
rf_bin_preds = rf_binary.predict(X_test_bin)

In [62]:
rf_bin_report = classification_report(y_test_bin, rf_bin_preds, zero_division=0)
rf_bin_cm = confusion_matrix(y_test_bin, rf_bin_preds)
rf_bin_acc = accuracy_score(y_test_bin, rf_bin_preds)

print("Binary RF Accuracy:", rf_bin_acc)
print(rf_bin_cm)
print(rf_bin_report)

Binary RF Accuracy: 0.9997839218661468
[[150874     18]
 [    27  57339]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    150892
           1       1.00      1.00      1.00     57366

    accuracy                           1.00    208258
   macro avg       1.00      1.00      1.00    208258
weighted avg       1.00      1.00      1.00    208258



In [63]:
rf_bin_probs = rf_binary.predict_proba(X_test_bin)[:, 1]
rf_bin_preds_tuned = (rf_bin_probs > 0.4).astype(int)

In [64]:
rf_bin_tuned_report = classification_report(y_test_bin, rf_bin_preds_tuned, zero_division=0)
rf_bin_tuned_cm = confusion_matrix(y_test_bin, rf_bin_preds_tuned)
rf_bin_tuned_acc = accuracy_score(y_test_bin, rf_bin_preds_tuned)

print("Tuned Binary RF Accuracy:", rf_bin_tuned_acc)
print(rf_bin_tuned_cm)
print(rf_bin_tuned_report)

Tuned Binary RF Accuracy: 0.9998127322839939
[[150873     19]
 [    20  57346]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    150892
           1       1.00      1.00      1.00     57366

    accuracy                           1.00    208258
   macro avg       1.00      1.00      1.00    208258
weighted avg       1.00      1.00      1.00    208258



In [65]:
scaler = StandardScaler()

X_train_bin_scaled = scaler.fit_transform(X_train_bin)
X_test_bin_scaled = scaler.transform(X_test_bin)

In [66]:
X_train_normal = X_train_bin_scaled[y_train_bin == 0]

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)

iso_model.fit(X_train_normal)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.1
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [67]:
iso_preds = iso_model.predict(X_test_bin_scaled)
iso_preds_binary = [0 if pred == 1 else 1 for pred in iso_preds]

In [68]:
iso_report = classification_report(y_test_bin, iso_preds_binary, zero_division=0)
iso_cm = confusion_matrix(y_test_bin, iso_preds_binary)
iso_acc = accuracy_score(y_test_bin, iso_preds_binary)

print("Isolation Forest Accuracy:", iso_acc)
print(iso_cm)
print(iso_report)

Isolation Forest Accuracy: 0.6823411345542548
[[135628  15264]
 [ 50891   6475]]
              precision    recall  f1-score   support

           0       0.73      0.90      0.80    150892
           1       0.30      0.11      0.16     57366

    accuracy                           0.68    208258
   macro avg       0.51      0.51      0.48    208258
weighted avg       0.61      0.68      0.63    208258



In [69]:
hybrid_preds = [
    1 if (rf == 1 or iso == 1) else 0
    for rf, iso in zip(rf_bin_preds_tuned, iso_preds_binary)
]

In [70]:
hybrid_report = classification_report(y_test_bin, hybrid_preds, zero_division=0)
hybrid_cm = confusion_matrix(y_test_bin, hybrid_preds)
hybrid_acc = accuracy_score(y_test_bin, hybrid_preds)

print("Hybrid Accuracy:", hybrid_acc)
print(hybrid_cm)
print(hybrid_report)

Hybrid Accuracy: 0.9265382362262194
[[135610  15282]
 [    17  57349]]
              precision    recall  f1-score   support

           0       1.00      0.90      0.95    150892
           1       0.79      1.00      0.88     57366

    accuracy                           0.93    208258
   macro avg       0.89      0.95      0.91    208258
weighted avg       0.94      0.93      0.93    208258



In [71]:
X_train_mc = df_train[features].copy()
y_train_mc = df_train["attack_type"].copy()

X_test_mc = df_test[features].copy()
y_test_mc = df_test["attack_type"].copy()

print("Multiclass train shape:", X_train_mc.shape)
print("Multiclass test shape:", X_test_mc.shape)

Multiclass train shape: (833030, 8)
Multiclass test shape: (208258, 8)


In [72]:
rf_attack_classifier = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_attack_classifier.fit(X_train_mc, y_train_mc)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [73]:
mc_preds = rf_attack_classifier.predict(X_test_mc)

In [74]:
mc_report = classification_report(y_test_mc, mc_preds, zero_division=0)
mc_cm = confusion_matrix(y_test_mc, mc_preds)
mc_acc = accuracy_score(y_test_mc, mc_preds)

print("Multiclass Accuracy:", mc_acc)
print(mc_cm)
print(mc_report)

Multiclass Accuracy: 0.999745507975684
[[150866     16     10]
 [    14  25591      0]
 [    10      3  31748]]
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00    150892
        DDoS       1.00      1.00      1.00     25605
    PortScan       1.00      1.00      1.00     31761

    accuracy                           1.00    208258
   macro avg       1.00      1.00      1.00    208258
weighted avg       1.00      1.00      1.00    208258



## Model Comparison

### Binary IDS Models
- Random Forest
- Tuned Random Forest
- Isolation Forest
- Hybrid Model

### Multiclass Attack Classifier
- Random Forest multiclass classifier

This comparison shows the difference between:
- binary intrusion detection
- anomaly detection
- attack-family classification


In [75]:
output_dir = "../outputs/general_model"
os.makedirs(output_dir, exist_ok=True)

In [76]:
with open(f"{output_dir}/binary_rf_results.txt", "w") as f:
    f.write(f"Binary RF Accuracy: {rf_bin_acc}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(rf_bin_cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(rf_bin_report)

In [77]:
with open(f"{output_dir}/binary_rf_tuned_results.txt", "w") as f:
    f.write(f"Tuned Binary RF Accuracy: {rf_bin_tuned_acc}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(rf_bin_tuned_cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(rf_bin_tuned_report)

In [78]:
with open(f"{output_dir}/binary_iso_results.txt", "w") as f:
    f.write(f"Isolation Forest Accuracy: {iso_acc}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(iso_cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(iso_report)

In [79]:
with open(f"{output_dir}/binary_hybrid_results.txt", "w") as f:
    f.write(f"Hybrid Accuracy: {hybrid_acc}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(hybrid_cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(hybrid_report)

In [80]:
with open(f"{output_dir}/attack_classifier_results.txt", "w") as f:
    f.write(f"Multiclass Accuracy: {mc_acc}\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(mc_cm) + "\n\n")
    f.write("Classification Report:\n")
    f.write(mc_report)

In [81]:
import os

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

In [82]:
joblib.dump(rf_binary, f"{model_dir}/general_binary_rf_model.pkl")

['../models/general_binary_rf_model.pkl']

In [83]:
joblib.dump(rf_attack_classifier, f"{model_dir}/general_attack_classifier.pkl")


['../models/general_attack_classifier.pkl']

In [84]:
joblib.dump(scaler, f"{model_dir}/general_scaler.pkl")

['../models/general_scaler.pkl']